### This Notebook is dedicated to Implementing Different Attention Mechanisms.

![Alt Text](pics/Attention_Implementation_Types.png)

> Decoder Only Transformer Architecture.

![Alt Text](pics/Decoder_Only_Transformer.png)

### 3.3.1 A simple self-attention mechanism without trainable weights

In [1]:
import torch

> Simple Attention Unnormalized

![Alt Text](pics/SimpleAttention.png)

In [2]:
#Dummy Sequence.
inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your (x^1)
[0.55, 0.87, 0.66], # journey (x^2)
[0.57, 0.85, 0.64], # starts (x^3)
[0.22, 0.58, 0.33], # with (x^4)
[0.77, 0.25, 0.10], # one (x^5)
[0.05, 0.80, 0.55]] # step (x^6)
)

print(f"Input Data Shape:{inputs.shape}")

Input Data Shape:torch.Size([6, 3])


In [7]:
#Implementing Simple Attention.
#Here we are going to calculate the simple un-normalized dot product attention with respect to the  journey token embedding.
query = inputs[1] #journey toke embedding
attention_scores_2 = torch.empty(size=(inputs.shape[0], )) #Since there are 6 token for which the query has to calculate attention score.
for i, x_i in enumerate(inputs):
    attention_scores_2[i] = torch.dot(input=x_i, tensor=query)

#The attention score of 2nd token(journey) w.r.t other tokens.
print(f'Attention Score of Journey token w.r.t other tokens:{attention_scores_2}')

Attention Score of Journey token w.r.t other tokens:tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


> Simple Attention Normalized

![Alt Text](pics/SimpleAttention_Normalized.png)

In [ ]:
#Simple Attention Normalization.
#But the Attention score of 2nd token w.r.t others was not normalized i.e sum of attention score/weights was not 1.
attention_scores_norm_2 = attention_scores_2 / attention_scores_2.sum()
print('Attention Weights Normalized:', attention_scores_norm_2)
print('Sum:', attention_scores_norm_2.sum())


Attention Weights Normalized: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


In [ ]:
#In practice, it's more common and advisable to use the softmax function for normalization.
#This approach is better at managing extreme values and offers more favorable gradient properties during training

def naive_softmax(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attention_scores_norm_2_naive = naive_softmax(attention_scores_2)
print('Attention Weights Normalized Naive Softmax:', attention_scores_norm_2_naive)
print('Sum:', attention_scores_norm_2_naive.sum())

Attention Weights Normalized Naive Softmax: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


In [15]:
#Now we will use the optimized pytorch softmax.
attention_scores_2_norm_torch = torch.softmax(input=attention_scores_2, dim=0)  #Summing across columns and here only 1 row.
print('Attention Weights Normalized Pytorch Softmax:', attention_scores_2_norm_torch)
print('Sum:', attention_scores_2_norm_torch.sum())

Attention Weights Normalized Pytorch Softmax: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


> Simple Attention Context Vector.

![Alt Text](pics/SimpleAttention_ContextVector.png)

In [13]:
torch.zeros(size=query.shape)

tensor([0., 0., 0.])

In [ ]:
#Context Vector from Simple Attention.
#SO Context vector is the simple attention score * token embedding.
query = inputs[1]
#Initializing a 0 vector to for adding up the context vector.
#We need to add the attention score * token embedding for each token so need a 0 vector to add up incrementally to get context vector 2 for 2nd token.
context_vector_2 = torch.zeros(size=query.shape) 
for i, x_i in enumerate(inputs):

    context_vector_2 += attention_scores_2_norm_torch[i] * inputs[i] #alpha_i * input_vector_i -> (1,) * (1,3) -> (1,3)

print(f"Context Vector for 2nd token is Token:Journey -> Context Vector:{context_vector_2}")

Context Vector for 2nd token is Token:Journey -> Context Vector:tensor([0.4419, 0.6515, 0.5683])


### 3.3.2 Computing attention weights for all input tokens

In [17]:
inputs.shape[0]

6

In [19]:
#Now we will calculate attention weights as well as the context vectors for all input tokens.
#Since there are 6 tokens with embedding size of 4 , so each token attend to 6 tokens so 36 attention scores under 6 x 6 self-attention matrix
attn_scores_vanilla = torch.empty(size=(inputs.shape[0], inputs.shape[0]))
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores_vanilla[i,j] = torch.dot(input=x_i, tensor=x_j)

print(f"The Whole Self Attention Matrix on the whole input sequence:\n{attn_scores_vanilla}")
print(f"Shape:{attn_scores_vanilla.shape}")

The Whole Self Attention Matrix on the whole input sequence:
tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])
Shape:torch.Size([6, 6])


In [21]:
inputs.shape

torch.Size([6, 3])

In [ ]:
#A easier way to get around this O(n^2) loop is using torch vector outer product.
attn_scores_optimized = inputs @ inputs.T  #Here @ is for matrix multiplication and inputs is shape(6,3) so inputs.T is shape:(3,6) i.e  it gives 6 x 6 matrix attention scores.
print(f"The Whole Self Attention Matrix on the whole input sequence:\n{attn_scores_optimized}")
print(f"Shape:{attn_scores_optimized.shape}")

The Whole Self Attention Matrix on the whole input sequence:
tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])
Shape:torch.Size([6, 6])


> In the above Whole Self - Attention Matrix row_i corresponds token_i attention w.r.t other tokens and so on.

> So 2nd row or row=1 means the attention score of journey token w.r.t other tokens.

In [22]:
#Now we need to Normalize the Self Attention Weights across columns i.e dim =1.
#Note: for 2d matrix dim = 0 -> row, dim = 1 -> column. for 3d dim=0 -> table, dim = 1 -> row, dim = 2 -> column
#Note: If we want to collapse a dimension we need to specify it.
#Like for Nomralization we sum across columns for each row collapse the col to 1 so we specify dim = 1 for a 2d matrix or dim =-1 for higher tensors

attn_scores_optimized_norm = torch.softmax(input=attn_scores_optimized, dim=1)
print(f"The Whole Self Attention Matrix on the whole input sequence:\n{attn_scores_optimized_norm}")
print(f"Shape:{attn_scores_optimized_norm.shape}")

The Whole Self Attention Matrix on the whole input sequence:
tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])
Shape:torch.Size([6, 6])


In [28]:
#Checking whether attention across rows sum to 1.
print(f"2nd token attention scores:{attn_scores_optimized_norm[[1,]]}")
print(f'Attention score sum for 2nd token:{attn_scores_optimized_norm[[1,]].sum(dim=1)}')

2nd token attention scores:tensor([[0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581]])
Attention score sum for 2nd token:tensor([1.])


In [30]:
#Now we need to calculate the context vectors for all 6 tokens.
#Once we have the attention_score shape(6,6) we just matrix multiply with inputs shape(6,3) giving us context vector shape(6,3)
#So attn_scores(6,6) @ inputs(6,3) -> context_vector(6,3)

all_context_vectors = attn_scores_optimized_norm @ inputs

print(f"Context Vectors for the whole 6 token Sequence:\n{all_context_vectors}")
print(f"Shape:{all_context_vectors.shape}")

Context Vectors for the whole 6 token Sequence:
tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])
Shape:torch.Size([6, 3])


### 3.4 Implementing self-attention with trainable weights

> Here we will introduce the W_Q, W_K, W_V trainables matrices for the Query, Key , Value transformation of the Input token vectors.

![Alt Text](pics/SelfAttention_Trainable.png)

In [2]:
#Dummy Sequence.
inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your (x^1)
[0.55, 0.87, 0.66], # journey (x^2)
[0.57, 0.85, 0.64], # starts (x^3)
[0.22, 0.58, 0.33], # with (x^4)
[0.77, 0.25, 0.10], # one (x^5)
[0.05, 0.80, 0.55]] # step (x^6)
)

print(f"Input Data Shape:{inputs.shape}")

Input Data Shape:torch.Size([6, 3])


In [3]:
#Here we will project the token embedding from 1x3 to 1x2 but in gpt's the dimension were kept same at the query, key, value level.
x_2 = inputs[1]
d_int = inputs.shape[1]   #Since there are 3 columns so the d_int is 3.
d_out = 2

print(f"x_2:{x_2}")

x_2:tensor([0.5500, 0.8700, 0.6600])


In [25]:
#Implementing the Attention for the journey token.
#Initializing the W_Q, W_K, W_V matrices.
#Here we will not use nn.Embedding which is a wrapper around base nn.Parameter with extra lookup method with indexing for token embedding fetching.
#nn.Parameter takes data as input, and since we need to train the matrices we initialize randomly.
#Also nn.Parameter takes requires_grad gradient calc during backprop but since it is a demo we are setting to false otherwise we need to set it to True
torch.manual_seed(123)
W_query = torch.nn.Parameter(data=torch.rand(size=(d_int, d_out)), requires_grad=False)
W_key = torch.nn.Parameter(data=torch.rand(size=(d_int, d_out)), requires_grad=False)
W_value = torch.nn.Parameter(data=torch.rand(size=(d_int, d_out)), requires_grad=False)

print(f"Query Matrix W_Q:{W_query}\n with Shape:{W_query.shape}")

Query Matrix W_Q:Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]])
 with Shape:torch.Size([3, 2])


In [11]:
#Now we are calculating the query_2, key_2, value_2 of the x_2 input. So for each of the we matrix multiply 1x3 & 3x2 giving us a projection of 1x2.
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

print(f"Query vector for x_2:{query_2}")
print(f"Key vector for x_2:{key_2}")
print(f"Value vector for x_2:{value_2}")

Query vector for x_2:tensor([0.4306, 1.4551])
Key vector for x_2:tensor([0.4433, 1.1419])
Value vector for x_2:tensor([0.3951, 1.0037])


In [12]:
#Now we calculate the Key & Value Vector for all tokens in the inputs sequnce which will give us a Key & Value matrix.
# inputs(6x3) @ W_key(3x2) -> Key Matrix (6x2)
keys = inputs @ W_key
values = inputs @ W_value

print(f"Keys for all the tokens in the Input Sequence:{keys}\n with shape:{keys.shape}")

Keys for all the tokens in the Input Sequence:tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]])
 with shape:torch.Size([6, 2])


> Attention Score for journey token w.r.t other tokens

![Alt Text](pics/SelfAttention_Trainable_AttentionScore.png)

In [ ]:
#Now we need to calculate the attention score for x_2 w.r.t all tokens keys.
#query_2(1x2) @ transpose(keys)(6x2) -> atten_scores_2(1x6)
#Later we also need to normalize the attention scores.
attn_scores_2 = query_2 @ keys.T
print(f"Attention Score vecxtor of x_2 w.r.t other inputs:{attn_scores_2}, \n with shape:{attn_scores_2.shape}")

Attention Score vecxtor of x_2 w.r.t other inputs:tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440]), 
 with shape:torch.Size([6])


> Now once we have the attention score we need to normalize it using softmax and also along the key vector dim i.e col here which is sqrt(d) (Here d=2, key dimension) that why it is scaled dot product attention called.

![Alt Text](pics/SelfAttention_Trainable_AttentionScore_Norm.png)

In [ ]:
#Now, we compute the attention weights by
# scaling the attention scores and using the softmax function we used earlier..
# The difference to earlier is that we now scale the attention scores by dividing
# them by the square root of the embedding dimension of the keys(Here d=2 as W_Q, W_key, W_value embeds to 1x2 dimension), (note that
# taking the square root is mathematically the same as exponentiating by 0.5):
#SO here two types of normalization ahppening one is across the inputs sequnce using softmax and another inside of softmax across the embedding dimension.
d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(input=attn_scores_2 / (d_k ** 0.5), dim=-1)
print(f"Normalized Attention Weights of x_2:{attn_weights_2}")

Normalized Attention Weights of x_2:tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


> Context Vector for X_2: SoftMax(Q @ K.T/sqrt(d)) @ V

![Alt Text](pics/SelfAttention_Trainable_ContextVector.png)

In [23]:
#Now we calculate the context vector for x_2.
#attn_weights_2(1x6) @ values(6x2) -> context_vector_2(1x2)
context_vector_2 = attn_weights_2 @ values
print(f"Context Vector for x_2 is :{context_vector_2}\n with shape:{context_vector_2.shape}")

Context Vector for x_2 is :tensor([0.3061, 0.8210])
 with shape:torch.Size([2])


> Now we will implement a Python Class for the Context Vector and Attention for all tokens instead of calculating for a single token

### 3.4.2 Implementing a compact self-attention Python class

> To get an idea of the difference between nn.Parameter vs nn.Liner vs nn.Embedding follow: https://audreywongkg.medium.com/pytorch-nn-parameter-vs-nn-linear-2131e319e463

> Here we will use nn.Module Class as Base Class around which we subclass our Self-Attention Class.

> In PyTorch, nn.Module is the base class for all neural network building blocks, meaning every model, layer, and connection you create must inherit from it to manage weights and compute gradients automatically.

What is nn.Module?Think of nn.Module as a smart blueprint for your neural network / layers and all. When you build a custom network class, you inherit from nn.Module. It provides two essential spaces: 

[1] __init__(self): The setup area where you define the structural components (like linear layers, convolutional layers, and pooling layers).forward(self, x): The operations area where you describe the actual path data takes as it travels forward through your network layers. 

[2] forward(self, x): The operations area where you describe the actual path data takes as it travels forward through your network layers. 

 By subclassing nn.Module, PyTorch automatically keeps track of all your trainable parameters (weights and biases), allows you to move the whole model to a GPU with .to('cuda'), and sets up backpropagation for training. 

![Alt Text](pics/SelfAttention_Trainable_Pipeline.png)

In [1]:
import torch
import torch.nn as nn

In [ ]:
#Implementing Self-Attention Version -1
class SelfAttention_V1(nn.Module):

    def __init__(self, d_in, d_out):  #d_in is the token embedding dimension & d_out is the our desired projection size.
        super().__init__()
        self.d_in, self.d_out = d_in, d_out
        self.W_q = nn.Parameter(torch.rand(size=(d_in, d_out)), requires_grad=True)  #W_q -> (d_in, d_out) #Here we are using rading checkpointing.
        self.W_k = nn.Parameter(torch.rand(size=(d_in, d_out)), requires_grad=True)    
        self.W_v = nn.Parameter(torch.rand(size=(d_in, d_out)), requires_grad=True)

    def forward(self, x):  #Here x is a matrix of input sequence so a matrix say example 6 tokens with each token of embedding dim of 3 so x - > (6x3) so then d_in = 3

        queries = x @ self.W_q  # (context_size, d_in) @ (d_in, d_out) -> (context_size, d_out)
        keys = x @ self.W_k     # (context_size, d_in) @ (d_in, d_out) -> (context_size, d_out)
        values = x @ self.W_v   # (context_size, d_in) @ (d_in, d_out) -> (context_size, d_out)
        #Attention Score is a Square Matrix.
        attn_scores = queries @ keys.T  #(context_size, d_out) @ transpose((context_size, d_out)) -> (context_size, context_size)
        #Nprmalizing the attention scores across Keys col dimension or d_out size and then applying softmax to normalize across the vocab size using dim=-1(last dim or col dim)
        attn_weights = torch.softmax(input=attn_scores / (keys.shape[-1])**0.5, dim=-1)
        #SO now for each token in vocab we get context vector of d_out size making a matrix of context_size, d_out.
        context_vectors = attn_weights @ values   # (context_size, context_size) @ (context_size, d_out) -> (context_size, d_out) 
        return  context_vectors

In [5]:
#Dummy Sequence.
inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your (x^1)
[0.55, 0.87, 0.66], # journey (x^2)
[0.57, 0.85, 0.64], # starts (x^3)
[0.22, 0.58, 0.33], # with (x^4)
[0.77, 0.25, 0.10], # one (x^5)
[0.05, 0.80, 0.55]] # step (x^6)
)

print(f"Input Data Shape:{inputs.shape}")

Input Data Shape:torch.Size([6, 3])


In [34]:
#Let's use the above Self-Attention Class.
torch.manual_seed(123)
sa_v1 = SelfAttention_V1(d_in=3, d_out=2)
context_vectors = sa_v1(inputs)
print(f"Context Vectors for all the tokens of the Input Sequence Matrix of {inputs.shape}:\n{context_vectors}")

Context Vectors for all the tokens of the Input Sequence Matrix of torch.Size([6, 3]):
tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


> Now we can also use nn.Liner instead of nn.Parameter for better matrices.

- Because in nn.Linear() uses better initialization strategies (Like:Xavier Initia) for W_q, W_k, W_v initialization unlike nn.Paramter where we need to pass the Initialized weights.

- Also nn.Linear() takes a input & when bias=False it works just like Matrix Multiplication internally.

- A significant advantage of using nn.Linear instead of manually implementing nn.Parameter(torch.rand(...)) is that nn.Linear has an optimized weight initialization scheme, contributing to more stable and effective model training.

In [3]:
#Implementing Self-Attention Version - 2
#In nn.Paramtere() if we use it to initialize W_q then to get queries we do inputs @ W_q 
#But in nn.Linear() if we initialize with the d_in, d_out the W_q and then just call W_q(inputs) it does the matrix multiplication internally with better weight initialization as well.
class SelfAttention_V2(nn.Module):
    
    def __init__(self, d_in, d_out):
        super().__init__()
        self.d_in, self.d_out = d_in, d_out
        #In nn.Linear we just need to give the input, output dim unlike nn.Parameters where we ahve give the randomly initialized data using torch.rand(size=(d_in, d_out))
        self.W_q = nn.Linear(in_features=d_in, out_features=d_out, bias=False)  #Here to replicate only matrix behavior we are setting bias=False
        self.W_k = nn.Linear(in_features=d_in, out_features=d_out, bias=False)
        self.W_v = nn.Linear(in_features=d_in, out_features=d_out, bias=False)

    def forward(self, x):

        Q, K, V = self.W_q(x), self.W_k(x), self.W_v(x)  #Here due to using nn.Linear() the x when passed to the layer it does the Matrix Multiplication on our behalf unlike nn.Parameter()
        attn_scores = Q @ K.T   
        attn_weights = torch.softmax(attn_scores/(K.shape[-1])**0.5, dim=-1)  #(context_size,context_size)
        context_vectors = attn_weights @ V  #(context_size, context_size) @ (context_size, d_out)
        return  context_vectors

In [6]:
torch.manual_seed(789)
sa_v2 = SelfAttention_V2(d_in=3, d_out=2)
context_vectors2 = sa_v2(inputs)
print(f"Context Vectors for all the tokens of the Input Sequence Matrix of {inputs.shape}:\n{context_vectors2}")

Context Vectors for all the tokens of the Input Sequence Matrix of torch.Size([6, 3]):
tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


In [42]:
sa_v2.W_q

Linear(in_features=3, out_features=2, bias=True)

In [46]:
sa_v2.W_q(inputs)

tensor([[ 0.6600, -0.2047],
        [ 0.9091, -0.4471],
        [ 0.8960, -0.4419],
        [ 0.5034, -0.2633],
        [ 0.4088, -0.2232],
        [ 0.6628, -0.3292]], grad_fn=<MmBackward0>)

### 3.5 Hiding future words with causal attention

![Alt Text](pics/CausalAttention.png)

> Now we will try to implement the Causal Attention.

- First we calculate the attention weights after Normalization 
- Then we zero out the upper traingular of the attention matrix like ones matrix (vocab,vocab) keeping the diagonal pat intact.
- Now we element wise multiply the Lower Triangular ones Matrix with Attention Weights Matrix to get the Causal/Mased Attention Score Matrix.
- Next we have again Normalize across the d_out dimension or dim=-1 or keys.shape[1] dimension to get the Causal/Mased Attention Weight Matrix

![Alt Text](pics/CausalAttention_Masked.png)

In [7]:
#Let's get the Q, K, V weights of the inputs.
#In pytorch one can access the attributes of the instance from outside as well unless it has been made private.
queries = sa_v2.W_q(inputs)
keys = sa_v2.W_k(inputs)
values = sa_v2.W_v(inputs)

attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores/(keys.shape[1]**0.5), dim=-1)
# context_vectors = attn_weights @ values


print(attn_weights)

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [8]:
#Now we need to create the masking.
#torch.tril gives lower traingular matrix for the given input matrix and diagonal thing control whether to return any diaginal or not.
mask_simple = torch.tril(input=torch.ones(size=(attn_weights.shape[0], attn_weights.shape[1])), diagonal=0)
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [9]:
#Now we element wise multiply the attn_weights with the Lower_Traingular_Ones Matrix
masked_attn_scores = attn_weights * mask_simple
print(masked_attn_scores)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


In [10]:
#Normalizing across the d_out embedding space to get Masked Attention Weight Matrix.
#Here keep_dim = True otherwise we would get a 1 x 6 instead of 6 x 1 vector and division /Normalization will be distorted.
masked_attn_weights = masked_attn_scores / masked_attn_scores.sum(dim=-1, keepdim=True)
print(masked_attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


In [11]:
# Now we get the context vector from the Masked Cross Attention Matrix.
causal_context_vectors = masked_attn_weights @ values
print(causal_context_vectors)

tensor([[-0.0872,  0.0286],
        [-0.0991,  0.0501],
        [-0.0999,  0.0633],
        [-0.0983,  0.0489],
        [-0.0514,  0.1098],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


> Now we implement an efficient version of the Causal Attention.

- Here we calculate the attention score and then direct do the upper triangular replacement with -inf.
- Here we just replace and fill the upper triangular part with -inf so when softmax is applied e^(-inf) = 0  happens directly.
- So Here we can avoid calculating the softmax for forbidden upper part directly because of e^(-inf) = 0 and also less operations.

![Alt Text](pics/CausalAttention_MaskedEfficient.png)

> Here we will use :

- torch.Tensor.masked_fill(mask, value) is a PyTorch function that fills elements of a tensor with a specified value where the mask is True. It returns a new tensor and does not change the original data unless you use its in-place counterpart, masked_fill_()

- Fills elements of self tensor with value where mask is True. The shape of mask must be broadcastable with the shape of the underlying tensor.

In [12]:
#Let's get the Q, K, V weights of the inputs.
#In pytorch one can access the attributes of the instance from outside as well unless it has been made private.
queries = sa_v2.W_q(inputs)
keys = sa_v2.W_k(inputs)
values = sa_v2.W_v(inputs)

attn_scores = queries @ keys.T
# attn_weights = torch.softmax(attn_scores/(keys.shape[1]**0.5), dim=-1)

#Efficient Implementation of Causal Attention.
masked_upper = torch.triu(input=torch.ones(size=(attn_weights.shape[0], attn_weights.shape[1])), diagonal=1)
causal_attn_scores_eff = attn_scores.masked_fill(masked_upper.bool(), -torch.inf)

print(causal_attn_scores_eff)

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


In [13]:
#Now we are using Softmax on this directly.
#Formula for Attention: Softmax(Q@K.T/sqrt(d))
causal_attn_weights = torch.softmax(input=causal_attn_scores_eff/(keys.shape[1]**0.5), dim=1)
print(causal_attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [14]:
#Context Vectors from Causal Attention.
#Formula for Context Vectors: Softmax(Q@K.T/sqrt(d)) @ V
context_vectors = causal_attn_weights @ values
print(context_vectors)

tensor([[-0.0872,  0.0286],
        [-0.0991,  0.0501],
        [-0.0999,  0.0633],
        [-0.0983,  0.0489],
        [-0.0514,  0.1098],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


### 3.5.2 Masking additional attention weights with dropout

![Alt Text](pics/CausalAttention_DropOut.png)

> ALthough after GPT-2 current llm models dont use DropOut commonly.

In [15]:
#When applying dropout to an attention weight matrix with a rate of 50%, half of the elements in the matrix are randomly set to zero. To compensate for the
# reduction in active elements, the values of the remaining elements in the matrix are scaled up by a factor of 1/0.5 =2. This scaling is crucial to
#maintain the overall balance of the attention weights, ensuring that the average influence of the attention mechanism remains consistent during both
#the training and inference phases.
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5) #A
example = torch.ones(6, 6) #B
print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [16]:
#Now we apply droput on the attention_weights.
torch.manual_seed(123)
dropout_causal = torch.nn.Dropout(0.5)
example_causal = dropout_causal(causal_attn_weights)

print(example_causal)

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)


### 3.5.3 Implementing a compact causal attention class

In [2]:
#Now we will implement Causal Attention Compact Class.
class CausalAttention(nn.Module):
    '''This class can handle batch of inputs so the Input can be (batch_size, seq_size, embed_dim_ize)'''
    def __init__(self, d_in, d_out, context_length, dropout=0.2, qkv_bias=False):
        # pass
        super().__init__()
        self.d_in, self.d_out = d_in, d_out
        self.context_length = context_length
        self.W_q = nn.Linear(in_features=d_in, out_features=d_out, bias=qkv_bias)
        self.W_k = nn.Linear(in_features=d_in, out_features=d_out, bias=qkv_bias)
        self.W_v = nn.Linear(in_features=d_in, out_features=d_out, bias=qkv_bias)

        #Now we are adding the Causal Attention Mask across out context length not the input sequnce size of current batch.
        #Context Length is = Max(Sequnce_Size) so it may happen that our model has varying input sizes across batches.
        #So we may group similar size input sequnce in same batches #
        #So a model with context_len = 1024 may have a batch with input size (8,6,3)->(batch_size, seq_size, embed_size) & also another batch may have (8,512,3)
        #But batch size is kept fixed.
        #So we create a general size causal mask of size context length.

        #Here we could ahve had self.mask = torch.triu(input=torch.ones(size=(context_length, context_length)), diagonal=1)
        #But during training if we do model.to_device('cuda') then the model is transported to cuda but this mask as injected from external is not considered part of the model so it may remain in the cpu.
        # So to avoid that problem here we are saying using register buffer that the tensor named 'mask' is part of the model. 
        self.register_buffer(name='mask', 
                             tensor=torch.triu(input=torch.ones(size=(context_length, context_length)), diagonal=1))
        self.dropout = nn.Dropout(p=dropout)


    def forward(self, x):

        batch_size, num_tokens, d_in = x.shape
        Q, K = self.W_q(x), self.W_k(x)  #Inside W_q/W_k Linear Layer : (batch_size, seq_size, d_in) @ (d_in, d_out) -> batch_size, seq_size, d_out
        V = self.W_v(x)
        #Pytorch supports batch multiplication where it internally treats in parallel batch_size_num of separate matrix multiplications.
        attn_scores = Q @ K.transpose(1, 2)  # (batch_size, seq_size, d_out) @ (batch_size, seq_size, d_out).transpose(dimension 1i.e row, dimension2 i.e col) -> (batch_size, seq_size, seq_size)

        #Now we apply the Causal Attention Mask in place so masked_fill_() instead of masked_fill()
        #Here also we apply a mask in only row, col shape but because of broadcasting it is applied to batch_size, row_size, col_size
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )
        attn_weights = torch.softmax(input=attn_scores/(K.shape[-1]**0.5), dim=-1)
        attn_weights = self.dropout(attn_weights)
        context_vectors = attn_weights @ V

        return  context_vectors

In [25]:
#Dummy Sequence.
inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your (x^1)
[0.55, 0.87, 0.66], # journey (x^2)
[0.57, 0.85, 0.64], # starts (x^3)
[0.22, 0.58, 0.33], # with (x^4)
[0.77, 0.25, 0.10], # one (x^5)
[0.05, 0.80, 0.55]] # step (x^6)
)

print(f"Input Data Shape:{inputs.shape}")

Input Data Shape:torch.Size([6, 3])


In [ ]:
#Stacking the inputs form demo example.
inputs

#torch.stack() concats two same shape tensors along a new dimension so if both are say (6,3) or (6,)/(6,1) then it makes them (2,6,3) or (2,6,1)
#torch.cat() concats along existing dimension and the concat dim only has to be same. So 2 (6,3) when cat across dim = 0  gives (12,3) and 2 (3,6) when torch.cat across dim=1 then (3,12)
#So for batch creation torch.stack and for multihead creation torch.cat is used generally.
batch = torch.stack((inputs, inputs), dim=0)  #Stacking new/across rows
print(batch.shape) 

torch.Size([2, 6, 3])


In [30]:
#Now let's initialize the Causal Attention Compact Class
torch.manual_seed(123)
context_length = batch.shape[1]
causal_attention = CausalAttention(d_in=batch.shape[2], d_out=2, context_length=context_length) #d_in is the initial_embedding_size of tokens.
context_vectors = causal_attention(batch)
print(f'Context Vector Generated using Causal Attention Mechanism:{context_vectors}')

Context Vector Generated using Causal Attention Mechanism:tensor([[[-0.5649,  0.2770],
         [-0.7343,  0.0072],
         [-0.7875, -0.0790],
         [-0.7093, -0.1053],
         [-0.6907, -0.1226],
         [-0.4311, -0.0610]],

        [[ 0.0000,  0.0000],
         [-0.7343,  0.0072],
         [-0.4833,  0.0046],
         [-0.7093, -0.1053],
         [-0.6907, -0.1226],
         [-0.5738, -0.1035]]], grad_fn=<UnsafeViewBackward0>)


In [31]:
context_vectors.shape

torch.Size([2, 6, 2])

### 3.6 Extending single-head attention to multi-head attention

![Alt Text](pics/Path_to_MultiHeadAttention.png)

> Naive Implementation of MultiHead Attention as Stacked Up Version of the Causal Attention.

![Alt Text](pics/MultiHeadAttention_Naive.png)

> Here we will use torch.nn.ModuleList(List[Layers]).

- If you use a standard Python list (e.g., self.layers = [nn.Linear(10, 10) for _ in range(5)]), PyTorch will be unaware of these layers. 

- As a result: The weights will not be included when you try to optimize your model (model.parameters() will throw an error or skip them).The layers will not move to the GPU when you run model.to('cuda').

- Also nn.Sequential, an nn.ModuleList does not have an internal forward method. You must explicitly iterate or index through it in your model's forward function.

In [3]:
#Here we will create a Wrapper named MultiHeadAttentionWrapper around Causal Attention Class.
class MultiHeadAttentionWrapper(nn.Module):
    """In this class we stack up multiple Causal Attention Heads."""
    def __init__(self, d_in, d_out, context_length, num_heads, dropout=0.2, qkv_bias=False):
        super().__init__()
        #torch.nn.ModuleList is a PyTorch container designed specifically to hold neural network submodules in a list
        self.heads = nn.ModuleList(modules=[CausalAttention(d_in, d_out, context_length, dropout=dropout, qkv_bias=qkv_bias) 
                                            for _ in range(num_heads)])

    def forward(self, x):
        #Here we first get the context vectors for each head and then concat across dim=1 or col dimension them not stack them as torch.cat()
        #gives us concatenation across existing dimension where torch.stack creates new dimension to stack.
        context_vectors = torch.cat([head(x) for head in self.heads], dim=-1)
        return  context_vectors

In [33]:
#Dummy Sequence.
inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your (x^1)
[0.55, 0.87, 0.66], # journey (x^2)
[0.57, 0.85, 0.64], # starts (x^3)
[0.22, 0.58, 0.33], # with (x^4)
[0.77, 0.25, 0.10], # one (x^5)
[0.05, 0.80, 0.55]] # step (x^6)
)

print(f"Input Data Shape:{inputs.shape}")

Input Data Shape:torch.Size([6, 3])


In [34]:
batch = torch.stack((inputs, inputs), dim=0)  #Stacking new/across rows
print(batch.shape) 

torch.Size([2, 6, 3])


In [38]:
#Here we initiating the Multiheaad Attention adn keep num_heads = 2.
torch.manual_seed(123)
multihead_attention = MultiHeadAttentionWrapper(d_in=batch.shape[-1], d_out=2, context_length=6, num_heads=2, dropout=0.0)
multihead_context_vectors = multihead_attention(batch)

print(f"MutiHead Context Vectors:{multihead_context_vectors} & \nshape :{multihead_context_vectors.shape}")

MutiHead Context Vectors:tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>) & 
shape :torch.Size([2, 6, 4])


- Here we can see that because of replicating the same Causal Attention block twice we got the context vector embedding size as twice the d_out

- We can also make it such that the d_out is never more than what is specified.

In [39]:
#Here we initiating the Multiheaad Attention adn keep num_heads = 2 but make sure the d_out is not twice rather exact as d_out=2.
torch.manual_seed(123)
context_embedding_out, num_heads = 2 , 2
d_out = int(context_embedding_out / num_heads)
multihead_attention = MultiHeadAttentionWrapper(d_in=batch.shape[-1], d_out=d_out, context_length=6, num_heads=num_heads, dropout=0.0)
multihead_context_vectors = multihead_attention(batch)

print(f"MutiHead Context Vectors:{multihead_context_vectors} & \nshape :{multihead_context_vectors.shape}")

MutiHead Context Vectors:tensor([[[-0.5740,  0.2216],
         [-0.7320,  0.0155],
         [-0.7774, -0.0546],
         [-0.6979, -0.0817],
         [-0.6538, -0.0957],
         [-0.6424, -0.1065]],

        [[-0.5740,  0.2216],
         [-0.7320,  0.0155],
         [-0.7774, -0.0546],
         [-0.6979, -0.0817],
         [-0.6538, -0.0957],
         [-0.6424, -0.1065]]], grad_fn=<CatBackward0>) & 
shape :torch.Size([2, 6, 2])


### 3.6.2 Implementing multi-head attention with weight splits & Better Efficient Implementation

> Here we will use torch.tensor.View() & torch.tensor.contiguous()

- view()
    - "Interpret the SAME memory with a different shape i.e can break or put together contiguous memory shapes."
    - So we can make a shape of say (batch, num_tokens, d_out) (8, 6, 32) -> (8, 6, 8, 4) (batch, num_tokens, num_heads, head_dim)
    - But Can't change directly the (batch, num_tokens, d_out) not possible -> (batch, num_heads, num_tokens,, head_dim). #For this we have to transpose it after View.

- transpose()/permute()
    - "Change the ORDER of dimensions."

- contiguous()
    - "Actually copy/rearrange the data so its
    - physical memory follows the current logical order."

- The Top section was our MultiHead Wrapper one which creates multiples Matrices for Multiple Heads .
- The Bottom section the efficient one which creates a giant Matrix and then breaks it to multiple matrices for Multiple Heads

![Alt Text](pics/MultiHeadAttention_Efficient.png)

In [ ]:
class MultiHeadAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length, batch_size, num_heads, dropout=0.2, qkv_bias=True):
        super().__init__()

        #Assert is for validation and if the condition doesn't hold throws Assertion Error with a message .
        #Syntax assert condition, <Message>
        assert d_out % num_heads == 0 , "d_out must be divisible by num_heads"
        self.head_dim = d_out // num_heads
        self.num_heads = num_heads
        self.d_in, self.d_out, self.context_length, self.batch_size = d_in, d_out, context_length, batch_size
        self.dropout = nn.Dropout(dropout)

        self.W_q = nn.Linear(in_features=d_in, out_features=d_out, bias=qkv_bias)
        self.W_k = nn.Linear(in_features=d_in, out_features=d_out, bias=qkv_bias)
        self.W_v = nn.Linear(in_features=d_in, out_features=d_out, bias=qkv_bias)

        self.register_buffer(name='mask',
                             tensor = torch.triu(input=torch.ones(size=(context_length, context_length)), diagonal=1)
                             )
        self.dropout = nn.Dropout(dropout)

        self.out_proj = nn.Linear(in_features=d_out, out_features=d_out)


    def forward(self, x):

        batch_size, num_tokens, d_in = x.shape  #shape(batch_size, num_tokens, input_embed_dim)

        #We initialize giant Q,K, V matrices and then break & distribute into multiple heads for efficint implementation.
        Q = self.W_q(x)   #Output Shape: (batch_size, num_tokens, d_out)
        K, V = self.W_k(x), self.W_v(x)

        #Now we need to break the d_out -> (num_heads, head_dim) using View()
        #So Q, K, V becomes from : (batch_size, num_tokens, d_out) -> (batch_size, num_tokens, num_heads, head_dim)
        Q = Q.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        K = K.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        V = V.view(batch_size, num_tokens, self.num_heads, self.head_dim)

        #Now we transpose the num_heads and num_tokens to later calculate attention and context vectors across separate Heads.
        Q = Q.transpose(1,2) # Input:(batch_size, num_tokens, num_heads, head_dim) ->  Output:(batch_size,  num_heads, num_tokens, head_dim)
        K = K.transpose(1,2)
        V = V.transpose(1,2)

        #The matrix multiplication implementation in PyTorch handles the 4-dimensional input tensor so that the matrix multiplication is carried out
        # between the 2 last dimensions (num_tokens, head_dim) and then repeated for the individual heads.
        #Here Pytorch handles the batch and multiple-head level matrxi multiplications by treating them multiple parallel matrix multiplications.
        # (batch_size,  num_heads, num_tokens, head_dim) @ (batch_size,  num_heads, head_dim, num_tokens) -> (batch_size,  num_heads, num_tokens, num_tokens)
        attn_scores = Q @ K.transpose(2,3) 

        # print(f'Attn_SCores:{attn_scores}\n')

        causal_mask = self.mask.bool()[:num_tokens, :num_tokens]
        #Inplace Masking.
        attn_scores.masked_fill_(mask=causal_mask, value=-torch.inf)
        #
        attn_weights = torch.softmax(attn_scores/(K.shape[-1]**0.5), dim=-1)
        attn_weights = self.dropout(attn_weights)
        # print(f'Attn_Weights:{attn_weights}\n')
        #(batch_size,  num_heads, num_tokens, num_tokens) @ (batch_size,  num_heads, num_tokens, head_dim) -> (batch_size,  num_heads, num_tokens, head_dim)
        context_vectors = attn_weights @ V 
        # print(f'contxt_vectors:{context_vectors}\n')


        #Now once we got the context for for all tokens across all heads we need to couple them.
        #For that we will transpose first to make num_heads & num_tokens dim change their place (batch_size,  num_heads, num_tokens, head_dim) -> (batch_size, num_tokens, num_heads, head_dim)
        context_vectors = context_vectors.transpose(1,2)

        #Now we make the context vectors current size as contiguous memory and then use View to couple the num_heads, head_dim to get d_out dim.
        #Input: (batch_size, num_tokens, num_heads, head_dim) -> (batch_size, num_tokens, d_out)
        context_vectors = context_vectors.contiguous().view(batch_size, num_tokens, self.d_out)
        context_vectors = self.out_proj(context_vectors)

        return  context_vectors

In [6]:
#Dummy Sequence.
inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your (x^1)
[0.55, 0.87, 0.66], # journey (x^2)
[0.57, 0.85, 0.64], # starts (x^3)
[0.22, 0.58, 0.33], # with (x^4)
[0.77, 0.25, 0.10], # one (x^5)
[0.05, 0.80, 0.55]] # step (x^6)
)

print(f"Input Data Shape:{inputs.shape}")
batch = torch.stack((inputs, inputs), dim=0)  #Stacking new/across rows
print(batch.shape) 

Input Data Shape:torch.Size([6, 3])
torch.Size([2, 6, 3])


In [18]:
d_in, context_length, batch_size = batch.shape[-1], 1024, batch.shape[0]

#In orginlal GPT-2 there was d_out = 768 , num_heads = 12, context length of 1024
#So each head_dim was 64 = 768//12
#Need to make sure that d_out is divisible by num_heads
d_out, num_heads= 16, 8
mha_head = MultiHeadAttention(d_in=d_in, d_out=d_out, context_length=context_length, batch_size=batch_size, num_heads=num_heads, dropout=0.0)

context_vectors = mha_head(batch)
print(f"Context Vectors: {context_vectors} \nShape:{context_vectors.shape}")

Context Vectors: tensor([[[ 0.3955,  0.2899, -0.1305, -0.3817, -0.0887,  0.8470, -0.2885,
          -0.3835,  0.3219, -0.3775, -0.2813, -0.3021, -0.4110,  0.6293,
          -0.3514,  0.0135],
         [ 0.5242,  0.4365, -0.1624, -0.3243, -0.1704,  0.8927, -0.2410,
          -0.3822,  0.5357, -0.4264,  0.0169, -0.0893, -0.4033,  0.5944,
          -0.4284,  0.0205],
         [ 0.5612,  0.4917, -0.1715, -0.3059, -0.1913,  0.9081, -0.2265,
          -0.3859,  0.6037, -0.4377,  0.0982, -0.0334, -0.3938,  0.5847,
          -0.4501,  0.0261],
         [ 0.5890,  0.5039, -0.1076, -0.2695, -0.1437,  0.8311, -0.1603,
          -0.3770,  0.5709, -0.4293,  0.1113,  0.0092, -0.3613,  0.5577,
          -0.4396,  0.0733],
         [ 0.5190,  0.5655, -0.0833, -0.2582, -0.0594,  0.8183, -0.1683,
          -0.4194,  0.5700, -0.3759,  0.1361,  0.0091, -0.2733,  0.5705,
          -0.3918,  0.1274],
         [ 0.5809,  0.5355, -0.0607, -0.2425, -0.0842,  0.7854, -0.1215,
          -0.3887,  0.5592, -0.4049

> GPT-2 style MHA dimension and size.

In [19]:
d_in, context_length, batch_size = batch.shape[-1], 1024, batch.shape[0]

d_out, num_heads= 768, 12
mha_head = MultiHeadAttention(d_in=d_in, d_out=d_out, context_length=context_length, batch_size=batch_size, num_heads=num_heads, dropout=0.0)

context_vectors = mha_head(batch)
print(f"Context Vectors: {context_vectors} \nShape:{context_vectors.shape}")

Context Vectors: tensor([[[-0.8045, -0.7630,  0.5328,  ...,  0.3633, -0.0426,  0.8955],
         [-0.7830, -0.8270,  0.5195,  ...,  0.1915,  0.0261,  0.7257],
         [-0.7746, -0.8403,  0.5072,  ...,  0.1321,  0.0585,  0.6639],
         [-0.7456, -0.8050,  0.5038,  ...,  0.1233,  0.0736,  0.5914],
         [-0.7173, -0.7310,  0.4080,  ...,  0.1706,  0.1652,  0.5890],
         [-0.7199, -0.7633,  0.4712,  ...,  0.1325,  0.1212,  0.5454]],

        [[-0.8045, -0.7630,  0.5328,  ...,  0.3633, -0.0426,  0.8955],
         [-0.7830, -0.8270,  0.5195,  ...,  0.1915,  0.0261,  0.7257],
         [-0.7746, -0.8403,  0.5072,  ...,  0.1321,  0.0585,  0.6639],
         [-0.7456, -0.8050,  0.5038,  ...,  0.1233,  0.0736,  0.5914],
         [-0.7173, -0.7310,  0.4080,  ...,  0.1706,  0.1652,  0.5890],
         [-0.7199, -0.7633,  0.4712,  ...,  0.1325,  0.1212,  0.5454]]],
       grad_fn=<ViewBackward0>) 
Shape:torch.Size([2, 6, 768])


> Here we can also implement Optimization of MHA like :
- GQA (Grouped-Query ATtention)
- MQA (Multi-QUery Attention)
- Flash Attention
- Paged Attention